# SpringGPT temporary model server

Turn on **Internet** and select a **GPU** in Kaggle Notebook settings. Run the cells in order. This starts the small Qwen3 0.6B model with thinking enabled and a temporary Cloudflare Quick Tunnel. Keep the last cell running.

The model and tunnel are public while running: do not send private information. This is a prototype, with one generation at a time and no guaranteed uptime. Kaggle GPU quotas and session limits apply.


In [ ]:
!pip -q install "transformers>=4.51,<5" "fastapi>=0.115,<1" "uvicorn>=0.30,<1"

In [ ]:
%%writefile springgpt_server.py
"""Temporary SpringGPT inference service for a Kaggle GPU notebook."""
import asyncio
import os
from contextlib import asynccontextmanager
from typing import Literal

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

MODEL = os.environ.get("SPRINGGPT_MODEL", "Qwen/Qwen3-0.6B")
MAX_NEW_TOKENS = int(os.environ.get("SPRINGGPT_MAX_NEW_TOKENS", "1024"))
model = None
tokenizer = None
lock = asyncio.Lock()

class Message(BaseModel):
    role: Literal["user", "assistant"]
    content: str = Field(min_length=1, max_length=4000)

class Chat(BaseModel):
    messages: list[Message] = Field(min_length=1, max_length=20)

@asynccontextmanager
async def lifespan(app):
    global tokenizer, model
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    tokenizer.truncation_side = "left"
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
    model.to("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    yield

app = FastAPI(lifespan=lifespan)

@app.get("/health")
def health():
    return {"ready": model is not None and tokenizer is not None}

def generate(messages):
    import torch
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    encoded = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(model.device)
    with torch.inference_mode():
        generated = model.generate(**encoded, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
                                   temperature=0.6, top_p=0.95, top_k=20,
                                   pad_token_id=tokenizer.eos_token_id)
    new_tokens = generated[0][encoded.input_ids.shape[1]:].tolist()
    # Qwen3 uses this token to mark the end of its thinking block.
    try:
        boundary = len(new_tokens) - new_tokens[::-1].index(151668)
    except ValueError:
        raise ValueError("The model did not finish thinking in the output budget")
    final = tokenizer.decode(new_tokens[boundary:], skip_special_tokens=True).strip()
    if not final:
        raise ValueError("The model returned no final answer")
    return final

@app.post("/chat")
async def chat(request: Chat):
    if model is None or tokenizer is None:
        raise HTTPException(status_code=503, detail="Model is starting")
    if lock.locked():
        raise HTTPException(status_code=429, detail="Model is busy")
    async with lock:
        try:
            result = await asyncio.to_thread(generate, [m.model_dump() for m in request.messages])
        except Exception:
            raise HTTPException(status_code=503, detail="The model could not finish its reply")
    return {"reply": result}


In [ ]:
import os, urllib.request
url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
urllib.request.urlretrieve(url, "/kaggle/working/cloudflared")
os.chmod("/kaggle/working/cloudflared", 0o755)
print("Tunnel client ready.")


In [ ]:
import subprocess, threading, re, time, sys

server = subprocess.Popen([sys.executable, "-m", "uvicorn", "springgpt_server:app", "--host", "127.0.0.1", "--port", "8000"])
tunnel = subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"], stderr=subprocess.PIPE, text=True, bufsize=1)

def show_tunnel():
    for line in tunnel.stderr:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            print("\nSpringGPT backend URL:", match.group(0), flush=True)
            print("Set backend_url to this value and enabled to true in the GitHub config file. Wait for the model to finish loading.", flush=True)

threading.Thread(target=show_tunnel, daemon=True).start()
print("Starting model and tunnel; keep this cell running.", flush=True)
try:
    while server.poll() is None and tunnel.poll() is None:
        time.sleep(2)
finally:
    server.terminate()
    tunnel.terminate()
    print("Server stopped.")
